# Import necessaries modules

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
#from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

# Define paths and other parameters

In [4]:
train_path = '/content/drive/My Drive/cat/train'
test_path = '/content/drive/My Drive/cat/test'
val_path = '/content/drive/My Drive/Cat/val'
img_height, img_width = 64, 64
batch_size = 32
num_epochs = 10

# Data transforms

In [7]:
transform = transforms.Compose([
    transforms.Resize((img_height, img_width)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load datasets

In [8]:
train_dataset = torchvision.datasets.ImageFolder(train_path, transform=transform)
test_dataset = torchvision.datasets.ImageFolder(test_path, transform=transform)
val_dataset = torchvision.datasets.ImageFolder(val_path, transform=transform)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/drive/My Drive/cat/train'

# Create data loaders

In [9]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

NameError: name 'train_dataset' is not defined

# Define the model

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3)
        self.conv3 = nn.Conv2d(64, 128, 3)
        self.fc1 = nn.Linear(128 * 6 * 6, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

model = CNN()


# Define loss function and optimizer

In [11]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# TensorBoard writer

In [12]:
log_dir = "logs/" + datetime.now().strftime("%Y%m%d-%H%M%S")
writer = SummaryWriter(log_dir)

NameError: name 'SummaryWriter' is not defined

# Training loop

In [13]:
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels.unsqueeze(1).float())
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:    # Print every 100 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / 100))
            running_loss = 0.0

    # Write loss to TensorBoard
    writer.add_scalar('Training Loss', running_loss, epoch)

print('Finished Training')

NameError: name 'train_loader' is not defined

# Evaluation

In [14]:
correct = 0
total = 0
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = model(images)
        predicted = torch.round(outputs)
        total += labels.size(0)
        correct += (predicted == labels.unsqueeze(1)).sum().item()

print('Accuracy of the network on the test images: %d %%' % (
    100 * correct / total))

NameError: name 'test_loader' is not defined

# Close TensorBoard writer

In [ ]:
writer.close()